In [15]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GEMINI_API_KEY']:
    print("GEMINI_API_KEY is set.")

GEMINI_API_KEY is set.


In [2]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

In [17]:
llm= ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                            api_key=os.environ['GEMINI_API_KEY'])

## RAG IMPLEMENTATION WITH PDF

# 
Step -1 
Extracting Text from PDF


In [19]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("databricks.pdf")
doc=loader.load()
doc

Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 92 0 (offset 0)
Ignoring wrong pointing object 112 0 (offset 0)
Ignoring wrong pointing object 130 0 (offset 0)
Ignoring wrong pointing object 148 0 (offset 0)
Ignoring wrong pointing object 166 0 (offset 0)
Ignoring wrong pointing object 185 0 (offset 0)
Ignoring wrong pointing object 203 0 (offset 0)
Ignoring wrong pointing object 221 0 (offset 0)
Ignoring wrong pointing object 381 0 (offset 0)
Ignoring wrong pointing object 403 0 (offset 0)
Ignoring wrong pointing object 582 0 (offset 0)
Ignoring wrong pointing object 624 0 (offset 0)
Ignoring wrong pointing object 646 0 (offset 0)
Ignoring wrong pointing object 665 0 (offset 0)
Ignoring wrong pointing object 684 0 (offset 0)
Ignoring wrong pointing object 710 0 (offset 0)


[Document(metadata={'producer': 'Mac OS X 10.13.6 Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20190109175909Z00'00'", 'title': 'DataFest_Jan2019_Databricks_Intro', 'moddate': "D:20190109175909Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': 'databricks.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1'}, page_content='Insight Presentation\nDatabricks, an IntroductionChuck Connell, Insight Digital Innovation'),
 Document(metadata={'producer': 'Mac OS X 10.13.6 Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20190109175909Z00'00'", 'title': 'DataFest_Jan2019_Databricks_Intro', 'moddate': "D:20190109175909Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': 'databricks.pdf', 'total_pages': 36, 'page': 1, 'page_label': '2'}, page_content='Speaker Bio•Senior Data Architect at Insight Digital Innovation•Focus on Azure big data services –HDInsight/Hadoop, Databricks, Cosmos DB •Related work…•NoSQL and relational data models, transitions•S

In [20]:
for i in doc:
    i.metadata={"source":"databricks.pdf",
                "author":"Microsoft"}
doc
    

[Document(metadata={'source': 'databricks.pdf', 'author': 'Microsoft'}, page_content='Insight Presentation\nDatabricks, an IntroductionChuck Connell, Insight Digital Innovation'),
 Document(metadata={'source': 'databricks.pdf', 'author': 'Microsoft'}, page_content='Speaker Bio•Senior Data Architect at Insight Digital Innovation•Focus on Azure big data services –HDInsight/Hadoop, Databricks, Cosmos DB •Related work…•NoSQL and relational data models, transitions•Size/volume estimates•JSON schemas for schema-less data•Boston office in Watertown Sq, on the river walk'),
 Document(metadata={'source': 'databricks.pdf', 'author': 'Microsoft'}, page_content='Creating meaningful connections that help businesses run smarter.\nSupply Chain OptimizationWe help you invest smarter so you can manage today and transform the future.\nConnected WorkforceWe create a connected workplace so employees can work smarter.\nCloud & Data Center TransformationWe help you prepare for the future and align workloads

# 
Step-2 Chunking

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [22]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks=splitter.split_documents(doc)
chunks
len(chunks)

36

#
Step-3 Embedding of chunks


In [23]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [24]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
      
)

#
Step-4 Store Embeddings in the Existing Vector DB

In [25]:
from langchain_community.vectorstores import Chroma


from langchain_community.vectorstores import Chroma
vectorstore=Chroma(persist_directory="./VectorDB", embedding_function=embeddings)


In [26]:
vectorstore.add_documents(chunks)

['3c1e9ca5-d20b-430e-b626-29e110e850c6',
 '48bb6555-8476-49c5-8fcf-1267315bc206',
 'd8541bb8-604a-4798-9b84-0b210ef35f84',
 'f7531acb-67d9-41c7-aebf-eeb07e5f9ad0',
 '63f28587-0517-4b85-b0fe-fd28020b80b0',
 'e6300a81-7fe1-489f-8978-184fb1c1b4c7',
 '54a609c0-45de-4e70-b261-a7923e28b6b1',
 '92bb0c2b-a8f4-43ef-a2a6-d5a82abb7047',
 'db977fa1-5e96-460e-b144-7a817d96c075',
 '96cd0feb-3ad1-4a38-81b1-b751a25f3fdd',
 'f433a746-71f5-45fc-9d2e-7a73dcc648ce',
 'ce8e7504-eb6a-479a-97e8-303f51be78a0',
 '59d7f11c-b12f-4b26-aa7a-897c80f719e2',
 '3c0497d5-fc4c-4179-91e6-5ff1fb83bc0e',
 'a77b00a5-3b87-4734-9f54-319fc8467cbe',
 '3de878ac-5889-4d5a-8743-d8eab6e3877a',
 'f95258a1-ed5e-4e18-b43c-00b928ce9a32',
 '7a8881e6-c981-4520-bddd-5d6c15c08768',
 'b1ccc247-9240-464c-a203-f0b8a503cf5f',
 '82699cff-01c6-4c71-a642-94b89cd9b71b',
 'e8d705f3-e372-4748-b610-596147b74a47',
 '8926f98e-9919-4fb5-a836-f07ae6e1ba4b',
 'a875a23d-fe2b-4ef0-8bbd-24ad9791eed5',
 '2b71b39e-2328-4c1e-9f79-2ce394cd50eb',
 '20a0cd2e-c6f8-

#
Step-5 Semantic Search

In [29]:
response=vectorstore.similarity_search("What is databricks?", k=3)
response



[Document(metadata={'author': 'Microsoft', 'source': 'databricks.pdf'}, page_content='Databricks•Databricks is a way to use Spark more conveniently •Databricks isSpark, but with a GUI and many automated features•Creation and configuration of server clusters•Auto-scaling and shutdown of clusters•Connections to various file systems and formats•Programming interfaces for Python, Scala, SQL, R•Integration with other Azure services•Available only as a cloud service, both Amazon and Azure•Let’s dive in…'),
 Document(metadata={'author': 'Microsoft', 'source': 'databricks.pdf'}, page_content='Start Databricks, Azure'),
 Document(metadata={'source': 'databricks.pdf', 'author': 'Microsoft'}, page_content='Start Databricks, Azure')]